# GoldenCheck Demo

**Data validation that discovers rules from your data.**

[![PyPI](https://img.shields.io/pypi/v/goldencheck?color=d4a017)](https://pypi.org/project/goldencheck/)
[![GitHub](https://img.shields.io/badge/github-goldencheck-black)](https://github.com/benzsevern/goldencheck)

This notebook demonstrates GoldenCheck's zero-config data validation pipeline.
No rules to write — just point it at your data.

In [ ]:
!pip install -q goldencheck

## 1. Create sample data with planted issues

In [ ]:
import os
import tempfile

import polars as pl

# Create a messy dataset with realistic data quality issues
data = {
    "id": list(range(1, 101)),
    "name": ["Alice", "Bob", "Charlie", None, "Eve"] * 20,
    "email": [
        "alice@example.com", "bob@example", "charlie@test.org",
        "not-an-email", "eve@company.co"
    ] * 20,
    "age": [28, 35, -5, 42, 999] * 20,
    "status": ["active", "inactive", "Active", "ACTIVE", "pending"] * 20,
    "signup_date": ["2024-01-15", "2024-02-20", "2024-03-10", "2024-04-05", "2024-05-01"] * 20,
    "last_login": ["2024-01-10", "2024-02-25", "2024-03-08", "2024-04-10", "2024-05-05"] * 20,
    "score": ["85", "92", "78", "high", "63"] * 20,
}

df = pl.DataFrame(data)
csv_path = os.path.join(tempfile.gettempdir(), "demo_data.csv")
df.write_csv(csv_path)

print(f"Created {csv_path} with {len(df)} rows, {len(df.columns)} columns")
df.head(10)

## 2. Scan the data

GoldenCheck discovers issues automatically — no config, no rules, no setup.

In [ ]:
from goldencheck.engine.scanner import scan_file
from goldencheck.engine.confidence import apply_confidence_downgrade

findings, profile = scan_file(csv_path)
findings = apply_confidence_downgrade(findings, llm_boost=False)

print(f"Found {len(findings)} issues")
profile

## 3. Explore findings

Each finding has severity, confidence, affected rows, and sample values.

In [ ]:
from goldencheck.models.finding import Severity

# Show errors and warnings (skip INFO)
important = [f for f in findings if f.severity >= Severity.WARNING]

for f in important:
    print(f"{'ERROR' if f.severity == Severity.ERROR else 'WARN ':5s}  "
          f"[{f.column}] {f.check}: {f.message}")

print(f"\n{len(important)} important findings out of {len(findings)} total")

## 4. Rich HTML display

In Jupyter, findings and profiles render as styled HTML tables.

In [ ]:
from goldencheck.notebook import ScanResult

# Wrap findings + profile for rich display
result = ScanResult(findings=findings, profile=profile)
result

## 5. Try your own data

Upload a CSV or point to a URL:

In [ ]:
# Option A: Upload a file in Colab
# from google.colab import files
# uploaded = files.upload()
# your_file = list(uploaded.keys())[0]

# Option B: Use a URL
# import urllib.request
# url = "https://raw.githubusercontent.com/datasets/airport-codes/main/data/airport-codes.csv"
# your_file = "/tmp/your_data.csv"
# urllib.request.urlretrieve(url, your_file)

# Then scan:
# findings, profile = scan_file(your_file)
# findings = apply_confidence_downgrade(findings, llm_boost=False)
# ScanResult(findings=findings, profile=profile)

## 6. CLI usage

GoldenCheck also works as a CLI tool:

In [ ]:
!goldencheck {csv_path} --no-tui

---

**Learn more:** [GitHub](https://github.com/benzsevern/goldencheck) | [PyPI](https://pypi.org/project/goldencheck/) | [Wiki](https://github.com/benzsevern/goldencheck/wiki)